<a href="https://colab.research.google.com/github/zwartepiet-hash/Young-Warrior-Logistics/blob/main/Warrior_Logistics_Intelligence_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fpdf gradio requests matplotlib pandas


In [ ]:
import requests, datetime, time, threading
import matplotlib.pyplot as plt
from fpdf import FPDF
import gradio as gr

# Konfiguráció
COINGECKO_URL = "https://coingecko.com"

# 1. PDF Generátor (Emoji-mentesített, stabil verzió)
def export_pdf(text, lang):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    clean_text = text.replace('$', 'USD ').replace('---', '').replace('💡', '')
    for line in clean_text.split('\n'):
        pdf.cell(200, 10, txt=line.encode('latin-1', 'replace').decode('latin-1'), ln=True)
    fname = f"Warrior_Report_{lang}.pdf"
    pdf.output(fname)
    return fname

# 2. Adatgyűjtő Őrszem (Thread-alapú háttérfolyamat)
def sentinel_collector():
    while True:
        try:
            # Csak csendben gyűjt, nem zavarja az API-t túl gyakran
            requests.get(COINGECKO_URL, timeout=10)
            time.sleep(60)
        except:
            time.sleep(10)

# Elindítjuk az Őrszemet a háttérben
threading.Thread(target=sentinel_collector, daemon=True).start()
print("🛡️ Az Őrszem aktív és figyeli a piacot.")


🛡️ Az Őrszem aktív és figyeli a piacot.


In [ ]:
import requests, datetime, time
import gradio as gr
import matplotlib.pyplot as plt
from fpdf import FPDF

# --- KONFIGURÁCIÓ ---

COINGECKO_URL = "https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd"

translations = {
    "Hungarian": {
        "title": "LOGISZTIKAI JELENTÉS", "base": "Alapdíj", "fuel": "Üzemanyag felár",
        "total": "Összesen", "date": "Dátum", "label_report": "Riport elemzés",
        "label_chart": "Költség Grafikon", "label_pdf": "PDF Mentése", "label_lang": "Nyelv",
        "market_info": "Piac stabil, a BTC árfolyama határozza meg a költségeket.",
        "gen_btn": "📊 JELENTÉS GENERÁLÁSA", "wait": "⚠️ API Korlát: Várj 1 percet!"
    },
    "English": {
        "title": "LOGISTICS REPORT", "base": "Base Fee", "fuel": "Fuel Surcharge",
        "total": "Total", "date": "Date", "label_report": "Report Analysis",
        "label_chart": "Cost Chart", "label_pdf": "Save PDF", "label_lang": "Language",
        "market_info": "Market is stable. BTC price drives dynamic costs.",
        "gen_btn": "📊 GENERATE REPORT", "wait": "⚠️ API Limit: Wait 1 minute!"
    }
}

# --- FUNKCIÓK ---
def check_auth(u, p):
    if u.lower() == "warrior" and p == "logisztika2026":
        return gr.update(visible=False), gr.update(visible=True)
    return gr.update(visible=True), gr.update(visible=False)

def get_analysis(lang):
    t = translations[lang]
    try:
        res = requests.get(COINGECKO_URL, headers={'User-Agent': 'Warrior-Bot/1.0'}, timeout=10)

        # API Hiba kezelése (429 - Too Many Requests)
        if res.status_code == 429:
            return t["wait"], None, gr.update(), gr.update(), gr.update(), gr.update(), gr.update()

        btc = res.json()['bitcoin']['usd']
        base, fuel = 150.0, btc / 1000

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar([t["base"], t["fuel"]], [base, fuel], color=['#3498db', '#e74c3c'])
        ax.set_title(f"BTC: ${btc}")

        report = f"--- {t['title']} ---\n{t['date']}: {datetime.datetime.now().strftime('%Y-%m-%d')}\nBTC: ${btc}\n{t['total']}: {base + fuel:.2f} EUR\n\nInfo: {t['market_info']}"

        # Visszatérés: Riport, Grafikon, Riport-Label, Grafikon-Label, PDF-Gomb, Lang-Label, GENERATE-Gomb
        return report, fig, gr.update(label=t["label_report"]), gr.update(label=t["label_chart"]), gr.update(value=f"📄 {t['label_pdf']}"), gr.update(label=t["label_lang"]), gr.update(value=t["gen_btn"])
    except:
        return "API Connection Error", None, gr.update(), gr.update(), gr.update(), gr.update(), gr.update()

def export_pdf(text, lang):
    pdf = FPDF(); pdf.add_page(); pdf.set_font("Arial", size=12)
    clean = text.replace('$', 'USD ').replace('---', '').replace('💡', '')
    for line in clean.split('\n'):
        pdf.cell(200, 10, txt=line.encode('latin-1', 'replace').decode('latin-1'), ln=True)
    fname = f"Warrior_Report_{lang}.pdf"; pdf.output(fname); return fname

# --- UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    with gr.Column() as login:
        u, p = gr.Textbox(label="User"), gr.Textbox(label="Pass", type="password")
        btn = gr.Button("🔓 LOGIN")

    with gr.Column(visible=False) as dash:
        lang = gr.Dropdown(choices=list(translations.keys()), value="Hungarian", label="Nyelv")
        gen = gr.Button("📊 GENERATE", variant="primary")
        with gr.Row():
            rep, cht = gr.Textbox(label="Riport", lines=8), gr.Plot(label="Grafikon")
        pdf_btn, pdf_out = gr.Button("📄 PDF"), gr.File(label="Download")

    btn.click(check_auth, [u, p], [login, dash])
    # FONTOS: 7 output van, hogy a gomb felirata is frissüljön!
    gen.click(get_analysis, lang, [rep, cht, rep, cht, pdf_btn, lang, gen])
    pdf_btn.click(export_pdf, [rep, lang], pdf_out)

demo.launch(share=True)


/tmp/ipykernel_15723/1030591748.py:64: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b19ef89965ed053b6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
